# Custom $B$-field
___
Created on 29. Apr. 2026 by Gregor Bock 

(0378 1735; ge27doc)

This code shall create a ```.npz``` file, such as the MSR returns it after measurement, to allow any custom B-field to be imputed to MSR to find a useful coil layup.

The working principle is:
- Specify B-field through function (B_x(x, y, z), B_y(x, y, z), B_z(x, y, z)) or specify vector potential (A_x(x, y, z), A_y(x, y, z), A_z(x, y, z))
- Use a background map of *your choice* to extract the background field and measurement points
- At each measurement point, superimpose the negative requested field
- Outbut the new array just like the original file in an ```.npz``` file.

## Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy
from pathlib import Path

import External_functions as fkt

## Read background field data

In [ ]:
Path_to_background = Path("D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Experiments\\background_01_2026-05-13_10-48-57\\map\\points")

shift_x = 0
shift_y = 0
shift_z = 0

target_point_coord = []
B_target_point_background = []

for filename in os.listdir(Path_to_background):                             # iterates through all documents in the points folder
    if filename.endswith(".npz"):                                           # only load the .npz files
        file_path = os.path.join(Path_to_background, filename)
        data = np.load(file_path)                                           # load the data of the current point

        x = data['x_mm'] * 1e-3 - shift_x                                   # Umrechnung in [m] und Versetzen des Nullpunkts
        y = data['y_mm'] * 1e-3 - shift_y
        z = data['z_mm'] * 1e-3 - shift_z

        if 'mean_Bx_pT' in data and 'mean_By_pT' in data and 'mean_Bz_pT' in data:

            mean_Bx = data['mean_Bx_pT'] * 1e-12                                # Umrechnen in [T]
            mean_By = data['mean_By_pT'] * 1e-12
            mean_Bz = data['mean_Bz_pT'] * 1e-12

        else:
            
            mean_Bx = np.nan                                                # These values will get interpolated later on!
            mean_By = np.nan
            mean_Bz = np.nan

            print(f'Warning: There are empty measurements which do not contain any B-field information except the measurement position at ({each_target_point}).')
        
        each_target_point = np.array([x, y, z]).T                           # (Npoints, 3)
        each_B_target = np.array([mean_Bx, mean_By, mean_Bz]).T             # (Npoints, 3)

        target_point_coord.append(each_target_point)
        B_target_point_background.append(each_B_target)

# Stack all files into final (total_Npoints, 3) arrays
target_point_coord = np.vstack(target_point_coord)                          # (total_Npoints, 3)
B_target_point_background = np.vstack(B_target_point_background)              # (total_Npoints, 3)




# Interpolate missing B-field values in the with-current measurements
missing_rows = np.any(np.isnan(B_target_point_background), axis=1)
if np.any(missing_rows):
    valid_rows = missing_rows == False
    valid_points = target_point_coord[valid_rows]
    missing_points = target_point_coord[missing_rows]
    valid_B = B_target_point_background[valid_rows]

    for comp in range(3):
        interpolated = scipy.interpolate.griddata(
            valid_points,
            valid_B[:, comp],
            missing_points,
            method = 'linear',
            fill_value = np.nan
        )

        if np.any(np.isnan(interpolated)):
            nearest = scipy.interpolate.griddata(
                valid_points,
                valid_B[:, comp],
                missing_points[np.isnan(interpolated)],
                method='nearest'
            )
            interpolated[np.isnan(interpolated)] = nearest

        B_target_point_background[missing_rows, comp] = interpolated

    print(f'Interpolated {missing_rows.sum()} missing B-field row(s) in with-current target data.')

## Plot background $B$-field

In [ ]:
x = target_point_coord[:,0]
y = target_point_coord[:,1]
z = target_point_coord[:,2]
Bnorm = np.linalg.norm(B_target_point_background, axis=1)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

scat = ax.scatter(x, y, z, c=Bnorm, s=100, cmap='viridis', alpha = 0.8)

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('z [m]')
ax.set_title('Magnetic Field Strength at Target Points')
cbar = fig.colorbar(scat, ax=ax, label='|B| [T]')
cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])

plt.show()

# Customize Fields!

**Make sure to input valid fields accourding to Maxwell equations!**

Field can be specified directly or through their vector potential $A$, although this is not recommended, due to few points and possibly high discretisation errors.

In [ ]:
Specify_B = True
Specify_A = False

def B(x, y, z):
    """B-field components (x, y, z coordinates)"""
    B_x = np.full_like(x, 0)
    B_y = 0.7 * 10**(-9) * y
    B_z = -0.7 * 10**(-9) * z
    
    return np.column_stack([B_x, B_y, B_z])  # Shape: (N_points, 3)

def A(x, y, z):
    """Vector potential for a uniform field in x-direction"""
    A_x = np.full_like(x, 0)
    A_y = np.full_like(y, 0)
    A_z = 0.7 * 10**(-9) * x
    
    return np.column_stack([A_x, A_y, A_z])  # Shape: (N_points, 3)

def curl(A_func, x, y, z):
    """
    Compute curl of vector potential using numerical differentiation.
    B = curl(A) = (∂A_z/∂y - ∂A_y/∂z, ∂A_x/∂z - ∂A_z/∂x, ∂A_y/∂x - ∂A_x/∂y)
    """
    h = 1e-6  # Step size for finite differences
    
    # Evaluate A at perturbed points
    A_center = A_func(x, y, z)
    A_dx = A_func(x + h, y, z)
    A_dy = A_func(x, y + h, z)
    A_dz = A_func(x, y, z + h)
    
    # Compute partial derivatives
    dA_dx = (A_dx - A_center) / h
    dA_dy = (A_dy - A_center) / h
    dA_dz = (A_dz - A_center) / h
    
    # curl(A) components
    B_x = dA_dy[:, 2] - dA_dz[:, 1]  # ∂A_z/∂y - ∂A_y/∂z
    B_y = dA_dz[:, 0] - dA_dx[:, 2]  # ∂A_x/∂z - ∂A_z/∂x
    B_z = dA_dx[:, 1] - dA_dy[:, 0]  # ∂A_y/∂x - ∂A_x/∂y
    
    return np.column_stack([B_x, B_y, B_z])

if Specify_B == True and Specify_A == False:
    B_des = B(x, y, z)

elif Specify_B == False and Specify_A == True:
    B_des = curl(A, x, y, z)

else:
    print('Specify exactly ONE field! Either magnetic-field $B$ OR magnetic-vector-potential $A$.')

## Compute $B$ for file considering the background field

$B_{input} = B_{background} - B_{desired}$

In [ ]:
B_input = B_target_point_background - B_des

## Plot $|B|$ in a scatter plot

In [ ]:
Bnorm = np.linalg.norm(B_input, axis=1)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

scat = ax.scatter(x, y, z, c=Bnorm, s=100, cmap='viridis', alpha = 0.8)

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('z [m]')
ax.set_title('Magnetic Field Strength at Target Points')
cbar = fig.colorbar(scat, ax=ax, label='|B| [T]')
cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])

plt.show()

## Save results in correct form

Directory containing a ```.npz``` file for each point.

In [ ]:
target_path = Path("D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Desired_map")

# Create directory if it doesn't exist
target_path.mkdir(parents=True, exist_ok=True)

# Convert units: meters to millimeters, Tesla to picotesla
B_in_pT = B_input * 10**(12)

# Save each point as a separate .npz file
num_points = target_point_coord.shape[0]
for i in range(num_points):
    # Extract coordinates for this point (convert m to mm)
    x_mm = target_point_coord[i, 0] * 1e3
    y_mm = target_point_coord[i, 1] * 1e3
    z_mm = target_point_coord[i, 2] * 1e3
    
    # Extract B-field values for this point (in pT)
    mean_Bx_pT = B_in_pT[i, 0]
    mean_By_pT = B_in_pT[i, 1]
    mean_Bz_pT = B_in_pT[i, 2]
    
    # Create filename (pad with zeros for consistency)
    filename = target_path / f"point_{i+1:03d}.npz"
    
    # Save as dictionary in .npz file
    np.savez(filename, 
             x_mm=x_mm, 
             y_mm=y_mm, 
             z_mm=z_mm, 
             mean_Bx_pT=mean_Bx_pT, 
             mean_By_pT=mean_By_pT, 
             mean_Bz_pT=mean_Bz_pT)

print(f"Saved {num_points} points to {target_path}")